# Advance Crime Data Pipeline

## Stage 2: Cleaning Layer

**Stakeholder:** Police Force Analytics Unit <br>
**Stage:** 2 of 5: Cleaning <br>
**Medallion Layer:** Silver 🥈 <br>
**Police Forces:** West Midlands · Thames Valley · Surrey · Dyfed-Powys <br>
**Authors:** Group 1 <br>
**Last Updated:** 19 May 2026 

This notebook is responsible for cleaning, validating, and standardising the raw Bronze data. All decisions about what to drop, impute, or type cast are documented below. The output is ready for further enrichment with population, deprivation, and house price data.

---
### Purpose

This notebook forms the **Silver layer** of the medallion pipeline. It is responsible for cleaning, validating, and standardising the raw Bronze data.

---

### How to Use

Running the notebook end to end. Run all cells in order from top to bottom. The notebook is designed to be fully repeatable. No manual steps are required between cells.

**Dropping more columns**

```python
cols_to_drop = ["context", "reported_by", "location" # Add more columns to drop here
]
crime = crime.drop(columns=cols_to_drop, errors="ignore")
```


## 1. Environment Setup

This section sets up the Snowflake and Python environment required for the pipeline. It creates the necessary **warehouse**, **database**, and **schemas** if not already exist.

In [ ]:
%%sql -r dataframe_2
-- Create a warehouse for compute
CREATE WAREHOUSE IF NOT EXISTS CRIME_WH
  WAREHOUSE_SIZE = 'X-SMALL'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE;

-- Create the database
CREATE DATABASE IF NOT EXISTS CRIME_PIPELINE;

-- Create schemas for each pipeline stage
CREATE SCHEMA IF NOT EXISTS CRIME_PIPELINE.RAW; -- Bronze
CREATE SCHEMA IF NOT EXISTS CRIME_PIPELINE.CLEAN; -- Silver
CREATE SCHEMA IF NOT EXISTS CRIME_PIPELINE.REPORTING; -- Gold

In [ ]:
import pandas as pd
from snowflake.snowpark.context import get_active_session
from snowflake.connector.pandas_tools import write_pandas

session = get_active_session()
session.sql("USE DATABASE CRIME_PIPELINE").collect()
session.sql("USE SCHEMA CLEAN").collect()

# Source and destination table references
BRONZE_TABLE = "CRIME_PIPELINE.RAW.BRONZE_CRIME_RAW"
SILVER_TABLE = "SILVER_CRIME_CLEAN"

# Confirm session is pointing at the correct database and schema
print("Session database :", session.get_current_database())
print("Session schema   :", session.get_current_schema())

## 2. Read from Bronze Table and Initial Inspection

A preliminary review of the Bronze (Raw) dataset to understand its structure, null distribution, and value ranges before any cleaning is undertaken.

In [ ]:
# Read the full Bronze table into pandas
crime = session.table(BRONZE_TABLE).to_pandas()

# Standardise column names: strip whitespace, lowercase, replace spaces with underscores
crime.columns = [c.strip().lower().replace(" ", "_") for c in crime.columns]

print("Bronze rows read :", len(crime))
print("Columns          :", crime.columns.tolist())

In [ ]:
# Visual inspection of the first five rows to confirm structure and raw values
crime.head()

In [ ]:
baseline_count = len(crime)
print("Baseline row count:", baseline_count)

In [ ]:
# Null counts across all columns -- informs decisions in Sections 3 and 4
null_summary = crime.isnull().sum().rename("null_count").to_frame()
null_summary["null_pct"] = (null_summary["null_count"] / len(crime) * 100).round(1)
print(null_summary)

## 3. Drop Columns

In this step, unnecessary columns are removed from the dataset to improve clarity and reduce the overall dataset size. This helps streamline downstream processing and ensures the data remains focused on fields required for reporting and analysis.

Three columns are dropped: <br>
1. **Context** <br>
2. **Reported by** <br>
3. **Location** <br>

**Context** column is 100% null and provides no analytical value, so it is removed from the dataset. <br>

**Reported by** column is dropped as the geographic area a crime occurs in is likely to affect house prices more than the police force that reported it. Therefore, this column is redundant.


In [ ]:
cols_to_drop = ["context", "reported_by", "location" #Add more columns to drop here
]
crime = crime.drop(columns=cols_to_drop, errors="ignore")

print("Columns dropped:", cols_to_drop)
print("Remaining columns:", crime.columns.tolist())

## 4. Handle Null Values

In this step, the dataset is checked for columns containing a high proportion of missing (null) values.

In [ ]:
null_fill_cols = [
    "crime_id",
    "last_outcome_category",
    "longitude",
    "latitude",
    "lsoa_code",
    "lsoa_name"
]

crime[null_fill_cols] = crime[null_fill_cols].fillna("NOT_RECORDED")

# Confirm no nulls remain in required reporting fields
remaining_nulls = crime.isnull().sum()
print("Remaining nulls per column:")
print(remaining_nulls[remaining_nulls > 0] if remaining_nulls.sum() > 0 else "None. All nulls resolved.")

## 5. Type Casting

All columns were ingested as strings in the Bronze layer. Appropriate types are applied here.


In [ ]:
# Month: cast to datetime
crime["month"] = pd.to_datetime(crime["month"], format="%Y-%m")

# Derive year and month_num for time-based analysis in Gold
crime["year"]      = crime["month"].dt.year
crime["month_num"] = crime["month"].dt.month

# Coordinates: cast to float where not suppressed
crime["longitude"] = pd.to_numeric(crime["longitude"], errors="coerce")
crime["latitude"]  = pd.to_numeric(crime["latitude"],  errors="coerce")

print("Dtypes after casting:")
print(crime.dtypes)

In [ ]:
# Normalise: strip whitespace and apply title case
crime["crime_type"] = crime["crime_type"].str.strip().str.title()

print("Distinct crime types after standardisation:")
print(crime["crime_type"].value_counts())

## 6. Duplicate Rows

Check for duplicate records at the reporting grain before cleaning to ensure crime counts are not inflated in the Gold layer.

In [ ]:
# Define the reporting grain -- the combination of columns that should uniquely identify each record
grain_cols = ["falls_within", "month", "crime_type", "crime_id"]

duplicate_count = crime.duplicated(subset=grain_cols).sum()
print(f"Duplicate records at reporting grain: {duplicate_count}")

if duplicate_count > 0:
    crime = crime.drop_duplicates(subset=grain_cols)
    print(f"Duplicates removed. Rows remaining: {len(crime)}")
else:
    print("No duplicates found — no action taken.")

## 7. Export to Silver Table



In [ ]:
cleaned_count = len(crime)
rows_removed  = baseline_count - cleaned_count

print("╔══════════════════════════════════════════════════════════════╗")
print("║            SILVER RECONCILIATION REPORT                     ║")
print("╚══════════════════════════════════════════════════════════════╝")
print(f"Baseline rows (Bronze) : {baseline_count:,}")
print(f"Rows after cleaning    : {cleaned_count:,}")
print(f"Rows removed           : {rows_removed:,}")
print(f"Retention rate         : {cleaned_count / baseline_count * 100:.1f}%")

print("\nRows by force:")
print(crime["falls_within"].value_counts())

print("\nRows by month:")
print(crime["month"].value_counts().sort_index())

In [ ]:
# Reset index before writing to suppress the non-standard index warning from write_pandas
crime = crime.reset_index(drop=True)

success, nchunks, nrows, _ = write_pandas(
    conn=session.connection,
    df=crime,
    table_name=SILVER_TABLE,
    database="CRIME_PIPELINE",
    schema="CLEAN",
    auto_create_table=True,  # creates table if it does not exist
    overwrite=True           # replaces existing data on each run
)

# Verify written row count matches cleaned row count
written_count = session.table(f"CRIME_PIPELINE.CLEAN.{SILVER_TABLE}").count()
assert written_count == cleaned_count, \
    f"Row count mismatch: {written_count} written vs {cleaned_count} expected"

print(f"Silver table written successfully")
print(f"Table  : CRIME_PIPELINE.CLEAN.{SILVER_TABLE}")
print(f"Rows   : {written_count:,}")

In [ ]:
crime = session.table(BRONZE_TABLE).to_pandas()
crime.columns = [c.strip().lower().replace(" ", "_") for c in crime.columns]
print(crime["falls_within"].unique())